# KURE 번아웃 — 이중 검증셋 평가 v1

**목적**: 현재 최고 모델(`stage2_model_syn_1to7.pt`)의 도메인 미스매치 정량화

## 평가 구성
| 검증셋 | 파일 | 건수 | 용도 |
|--------|------|------|------|
| 구어체 | `stage2_val_v3.csv` | 4,395 | 기존 실험과 비교 (베이스라인) |
| 일기체 | `stage2_val_diary.csv` | 2,803 | 실제 사용 환경 근사 |

## 산출물
1. 두 검증셋에서의 F1, Accuracy, Classification Report
2. Confusion Matrix (각 검증셋별)
3. 오류 케이스 샘플 (어느 카테고리 → 어느 카테고리로 혼동되는지)

> F1 격차(구어체 vs 일기체) = 도메인 미스매치의 실험적 근거

## 1. 환경 설정 & Imports

In [ ]:
import os, warnings, copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. 경로 & 카테고리 매핑

In [ ]:
DATA_PATH  = 'D:/Programming/Projects/Burnout/llm/dataset'
MODEL_PATH = 'D:/Programming/Projects/Burnout/llm/models'

STAGE2_CATEGORIES = {0: '정서적_고갈', 1: '좌절_압박', 2: '부정적_대인관계', 3: '자기비하'}
CATEGORY_LIST     = list(STAGE2_CATEGORIES.values())

VAL_COLLOQUIAL = f'{DATA_PATH}/stage2_val_v3.csv'
VAL_DIARY      = f'{DATA_PATH}/stage2_val_diary.csv'

MODEL_SYN_1TO7 = f'{MODEL_PATH}/stage2_model_syn_1to7.pt'

MAX_LEN = 128
BATCH_SIZE = 64

print('[경로 확인]')
for name, path in [('Val 구어체', VAL_COLLOQUIAL), ('Val 일기체', VAL_DIARY),
                   ('Model 1:7',  MODEL_SYN_1TO7)]:
    mark = '✅' if os.path.exists(path) else '❌'
    print(f'  {mark} {name:<12} {path}')

## 3. 모델 구조 정의 (v3 노트북과 동일)

In [ ]:
class E2EBurnoutModel(nn.Module):
    def __init__(self, backbone, hidden_dim=256, num_classes=4, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(1024, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def mean_pool(self, token_embeds, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (token_embeds * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        return self.classifier(pooled)

## 4. KURE 로드 & 모델 빌드 헬퍼

In [ ]:
print('KURE 로딩 중...')
st_model      = SentenceTransformer('nlpai-lab/KURE-v1')
tokenizer     = st_model.tokenizer
backbone_orig = st_model[0].auto_model
del st_model
torch.cuda.empty_cache()
print('✅ KURE 로드 완료')

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True,
                     max_length=MAX_LEN, return_tensors='pt')

def load_eval_model(model_path: str) -> tuple:
    """평가 전용 모델 로드. backbone 복사 → state_dict 덮어쓰기."""
    backbone = copy.deepcopy(backbone_orig)
    for p in backbone.parameters():
        p.requires_grad = False
    model = E2EBurnoutModel(backbone).to(device)
    ckpt  = torch.load(model_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, ckpt

## 5. 검증셋 로드 & 토크나이징

In [ ]:
def build_loader(csv_path: str) -> tuple:
    df = pd.read_csv(csv_path)
    enc = tokenize(df['text'].tolist())
    labels = torch.tensor(df['label'].values, dtype=torch.long)
    ds = TensorDataset(enc['input_ids'], enc['attention_mask'], labels)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    return df, loader

print('[구어체 val]')
df_colloq, loader_colloq = build_loader(VAL_COLLOQUIAL)
print(f'  {len(df_colloq):,}건, 분포: {df_colloq["label"].value_counts().to_dict()}')

print('\n[일기체 val]')
df_diary, loader_diary = build_loader(VAL_DIARY)
print(f'  {len(df_diary):,}건, 분포: {df_diary["label"].value_counts().to_dict()}')

## 6. 평가 함수

In [ ]:
def evaluate(model, loader) -> tuple:
    all_preds, all_labels = [], []
    with torch.no_grad():
        for input_ids, attn, lbl in loader:
            with autocast():
                logits = model(input_ids.to(device), attn.to(device))
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(lbl.tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return np.array(all_preds), np.array(all_labels), acc, f1

## 7. 최고 모델 (Synthetic v3 1:7) 평가

In [ ]:
print('모델 로드 중: stage2_model_syn_1to7.pt')
model, ckpt = load_eval_model(MODEL_SYN_1TO7)
print(f'  Training best F1 (at save time): {ckpt["best_metrics"]["f1"]:.4f}')
print(f'  Data version: {ckpt.get("data_version", "N/A")}')

print('\n[구어체 val 평가]')
preds_c, labels_c, acc_c, f1_c = evaluate(model, loader_colloq)
print(f'  Accuracy: {acc_c:.4f}  |  Macro F1: {f1_c:.4f}')
print(classification_report(labels_c, preds_c,
      target_names=CATEGORY_LIST, digits=4))

print('\n[일기체 val 평가]')
preds_d, labels_d, acc_d, f1_d = evaluate(model, loader_diary)
print(f'  Accuracy: {acc_d:.4f}  |  Macro F1: {f1_d:.4f}')
print(classification_report(labels_d, preds_d,
      target_names=CATEGORY_LIST, digits=4))

print('\n' + '='*50)
print(f'  구어체 F1: {f1_c:.4f}')
print(f'  일기체 F1: {f1_d:.4f}')
print(f'  격차     : {f1_d - f1_c:+.4f}  ({"일기체 우위" if f1_d > f1_c else "구어체 우위"})')
print('='*50)

## 8. Confusion Matrix

두 검증셋 모두에 대해 시각화. **어느 카테고리끼리 섞이는지**가 핵심 정보.

In [ ]:
def plot_confusion(labels, preds, title: str, ax):
    cm = confusion_matrix(labels, preds, labels=[0,1,2,3])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalized (recall)

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(CATEGORY_LIST, rotation=30, ha='right')
    ax.set_yticklabels(CATEGORY_LIST)
    ax.set_xlabel('예측'); ax.set_ylabel('정답')
    ax.set_title(title)

    for i in range(4):
        for j in range(4):
            color = 'white' if cm_norm[i,j] > 0.5 else 'black'
            ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                    ha='center', va='center', color=color, fontsize=9)
    return im

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_confusion(labels_c, preds_c, f'구어체 val (F1={f1_c:.4f})', axes[0])
plot_confusion(labels_d, preds_d, f'일기체 val (F1={f1_d:.4f})', axes[1])
plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/../docs/confusion_matrix_eval_v1.png', dpi=120, bbox_inches='tight')
plt.show()
print('저장: docs/confusion_matrix_eval_v1.png')

## 9. 오류 케이스 샘플

각 (정답, 예측) 조합에서 틀린 케이스 최대 3개씩 추출 → 라벨 품질 vs 모델 한계 구분용.

In [ ]:
def show_error_samples(df: pd.DataFrame, preds: np.ndarray, labels: np.ndarray,
                       title: str, n_per_pair: int = 3):
    print('='*80)
    print(title)
    print('='*80)
    for true_idx in range(4):
        for pred_idx in range(4):
            if true_idx == pred_idx:
                continue
            mask = (labels == true_idx) & (preds == pred_idx)
            if mask.sum() == 0:
                continue
            indices = np.where(mask)[0][:n_per_pair]
            print(f'\n[정답: {CATEGORY_LIST[true_idx]:<10} → 예측: {CATEGORY_LIST[pred_idx]:<10}]'
                  f'  (총 {mask.sum()}건)')
            for idx in indices:
                text = df.iloc[idx]['text']
                text_preview = text[:80] + ('...' if len(text) > 80 else '')
                print(f'  - {text_preview}')

show_error_samples(df_colloq, preds_c, labels_c, '구어체 val 오류 샘플')
show_error_samples(df_diary,  preds_d, labels_d, '일기체 val 오류 샘플')

## 10. 결과 요약 (기록용)

아래 출력을 `CHANGES.md`에 반영할 것.

In [ ]:
summary = {
    'model': 'stage2_model_syn_1to7.pt',
    'colloquial_val': {
        'n': len(labels_c),
        'accuracy': round(acc_c, 4),
        'f1_macro': round(f1_c, 4),
    },
    'diary_val': {
        'n': len(labels_d),
        'accuracy': round(acc_d, 4),
        'f1_macro': round(f1_d, 4),
    },
    'gap_f1': round(f1_d - f1_c, 4),
}

import json
print(json.dumps(summary, indent=2, ensure_ascii=False))